# 01. LangChain 기초


> LangChain 언어 모델에 기반한 AI 애플리케이션을 쉽게 개발할 수 있도록 도와주는 프레임워크
>> 기존에 오픈AI와 같은 언어 모델의 API를 사용해 원하는 기능을 구현하려면 모든 코드를 직접 작성해야 했으나, 랭체인은 이 작업을 간소화할 수 있는 다양한 도구와 모듈 제공

>  **OpenAI** `.env`의 `OPENAI_API_KEY` 그대로 사용

---
## 0. 설치

In [2]:
!pip install openai python-dotenv langchain langchain-openai langchain-core

---
## 1. Day 3 OpenAI SDK vs LangChain

| | Day 3 (OpenAI SDK) | Day 5 (LangChain) |
|---|---|---|
| 호출 | `client.chat.completions.create()` | `llm.invoke()` |
| 메시지 | `{'role':'user','content':...}` | `HumanMessage`, `SystemMessage` |
| 체인 | 직접 함수 조합 | **LCEL** (`\|` 파이프) |
| RAG/Agent | 직접 루프 작성 | Retriever · AgentExecutor |

---
## 2. ChatOpenAI 연결

In [3]:
from pathlib import Path
from dotenv import load_dotenv

# load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1) #기존 파이썬 코드랑 다르게 API_KEY를 따로 설정 안해줘도 됨

# 문자열만 넘겨도 됨
out = llm.invoke('LangChain이 뭐야? 한 문장으로.') #invoke로 질문할 수 있음

In [5]:
out

AIMessage(content='LangChain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 18, 'total_tokens': 51, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42e6cfce1b', 'id': 'chatcmpl-DutPLbEhXyX1xMeAIX9PGGrsshXZ7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f0266-e768-79a2-a7c4-bbf465d712bb-0', usage_metadata={'input_tokens': 18, 'output_tokens': 33, 'total_tokens': 51, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
out.content

'LangChain은 자연어 처리(NLP) 모델을 활용하여 다양한 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.'

In [7]:
#정석
#사람이 보내는 메세지
from langchain_core.messages import HumanMessage  
input_prompt = HumanMessage(content='아무거나 쓰기')
message = [HumanMessage(content = "Langchain이 뭐야? 한 문장으로.")] #메세지는 list이므로 계속 늘려가는형태
out = llm.invoke(message)

In [ ]:
out

In [ ]:
# message에 멀티턴 하는 방법
message.append(out)

---
## 3. Message 타입 (Day 3 messages 리스트와 동일 역할)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 기존에 사용했던 방법 : system, user 별 content를 짜주는 것
# messages = [
#     {"role": "system", "content": '너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'},  # 초기 시스템 메시지
#     {"role": "user", "content" : 'RAG가 뭐야? 2문장으로 설명해.'} # 사용자 메시지
# ]

messages = [
    SystemMessage(content='너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'), # 초기 시스템 메시지
    HumanMessage(content='RAG가 뭐야? 2문장으로 설명해.'), # 사용자 메시지
]


In [9]:
# def get_ai_response(messages):
#     response = client.chat.completions.create(
#         model="gpt-4o",  # 응답 생성에 사용할 모델 지정
#         temperature=0.9,  # 응답 생성에 사용할 temperature 설정
#         messages=messages,  # 대화 기록을 입력으로 전달
#     )
#     return response.choices[0].message.content  # 생성된 응답의 내용 반환

# answer = get_ai_response(messages)


answer = llm.invoke(messages)
print(answer.content)
print('타입:', type(answer))

RAG는 "Retrieval-Augmented Generation"의 약자로, 정보 검색과 생성 모델을 결합한 기술입니다. 이 방법은 외부 데이터베이스에서 정보를 검색하여 더 정확하고 풍부한 응답을 생성하는 데 사용됩니다.
타입: <class 'langchain_core.messages.ai.AIMessage'>


In [12]:
answer
print(type(answer))

<class 'langchain_core.messages.ai.AIMessage'>


멀티턴 실습 lanchain_multiturn.py : 기존 open ai api -> langchain으로 교체

---
## 4. 메시지 히스토리 (멀티턴)

기존에는 멀티턴 대화를 위해 사용자의 대화 내용을 리스트나 딕셔너리에 추가하는 코드 작성

-> 메시지 히스토리 (Message History) 기능으로 쉽게 구현 가능

In [13]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4o-mini")

In [14]:
store = {}

In [ ]:
# 세션별 대화 기록을 저장할 딕셔너리
store = {}

In [15]:
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

In [ ]:
# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history) #history 저장하면서 model 작동

c:\Users\Admin\Miniconda3\envs\day4\lib\site-packages\IPython\core\interactiveshell.py:3550: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [17]:
store

{}

In [18]:
with_message_history

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x000002A1B3E13F70>, history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [19]:
config = {"configurable": {"session_id": "first"}}  # 세션 ID를 설정하는 config 객체 생성 #key : configurable, value : session_id를 key로 가지는 dict

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 신범교 라고 해.")],
    config=config,
)

print(response.content)

안녕, 신범교 님! 만나서 반가워요. 어떻게 도와드릴까요?


In [20]:
store

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕, 신범교 님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 18, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-DutmpIgY4mo5K6QCD7M7O02ixmhUW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-20df-7343-b064-5ba1e5acbc5a-0', usage_metadata={'input_tokens': 18, 'output_tokens': 23, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])}

In [21]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕, 신범교 님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 18, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-DutmpIgY4mo5K6QCD7M7O02ixmhUW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-20df-7343-b064-5ba1e5acbc5a-0', usage_metadata={'input_tokens': 18, 'output_tokens': 23, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [22]:
get_session_history('first')

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕, 신범교 님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 18, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-DutmpIgY4mo5K6QCD7M7O02ixmhUW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-20df-7343-b064-5ba1e5acbc5a-0', usage_metadata={'input_tokens': 18, 'output_tokens': 23, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [28]:
config = {"configurable": {"session_id": "third"}}  # 세션 ID를 설정하는 config 객체 생성 #key : configurable, value : session_id를 key로 가지는 dict

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

죄송하지만, 제가 알 수 있는 정보가 없습니다. 당신의 이름을 알려주시면 그에 맞춰 대화할 수 있을 것 같습니다.


In [23]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

#store에 있는 값을 가져오고 다음 질문에 대한걸 다시 저장

당신의 이름은 신범교입니다. 다른 질문이나 궁금한 점이 있으면 말씀해 주세요!


In [31]:
print(store['third'].messages)
print(store['second'].messages)

[HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}), AIMessage(content='죄송하지만, 제가 알 수 있는 정보가 없습니다. 당신의 이름을 알려주시면 그에 맞춰 대화할 수 있을 것 같습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 13, 'total_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_303afd94e3', 'id': 'chatcmpl-DutsY1TT634KaycCkGHQoZNCrjh9w', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f0282-8eed-74b3-9dd3-62962ce8c320-0', usage_metadata={'input_tokens': 13, 'output_tokens': 31, 'total_tokens': 44, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]
[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwar

In [33]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕, 신범교 님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 18, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-DutmpIgY4mo5K6QCD7M7O02ixmhUW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-20df-7343-b064-5ba1e5acbc5a-0', usage_metadata={'input_tokens': 18, 'output_tokens': 23, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐지?', additional_

In [34]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕, 신범교 님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 18, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-DutmpIgY4mo5K6QCD7M7O02ixmhUW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-20df-7343-b064-5ba1e5acbc5a-0', usage_metadata={'input_tokens': 18, 'output_tokens': 23, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}),


In [35]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 다른 질문이 있거나 더 이야기하고 싶은 내용이 있다면 말씀해 주세요!


In [36]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕, 신범교 님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 18, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-DutmpIgY4mo5K6QCD7M7O02ixmhUW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-20df-7343-b064-5ba1e5acbc5a-0', usage_metadata={'input_tokens': 18, 'output_tokens': 23, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}),


In [37]:
config = {"configurable": {"session_id": "second"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 맞나요? 다른 질문이나 도움이 필요하시면 말씀해 주세요!


In [38]:
store

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕, 신범교 님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 18, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-DutmpIgY4mo5K6QCD7M7O02ixmhUW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-20df-7343-b064-5ba1e5acbc5a-0', usage_metadata={'input_tokens': 18, 'output_tokens': 23, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐지?', a

In [39]:
config = {"configurable": {"session_id": "first"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 다른 도움이 필요하신가요?


In [40]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕, 신범교 님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 18, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-DutmpIgY4mo5K6QCD7M7O02ixmhUW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027d-20df-7343-b064-5ba1e5acbc5a-0', usage_metadata={'input_tokens': 18, 'output_tokens': 23, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}),


In [41]:
# 스트림 방식으로 출력
config = {"configurable": {"session_id": "first"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내 이름이 뭔지 말하고 이름의 뜻을 유추해서 말해봐 ")],
    config=config):
    print(r.content, end="") # 


당신의 이름은 "신범교"입니다. 이름의 각 부분을 살펴보면:

- **신(新)**: 일반적으로 "새롭다"는 의미를 가지고 있습니다.
- **범(範)**: "본보기가 되다"라는 뜻을 지니고 있으며, 또한 "범"이라는 동물이 강함과 권위를 상징하기도 합니다.
- **교(敎)**: "가르침" 또는 "교훈"의 의미를 가지고 있습니다.

이런 의미를 종합해보면, "신범교"라는 이름은 "새로운 본보기가 되는 가르침" 혹은 "강한 가르침을 전하는 새로운 성격"이라는 뜻으로 해석할 수 있습니다. 이름에 어떤 특별한 의미나 배경이 있는지도 이야기해 주실 수 있으면 좋겠습니다!

---
## 5. LCEL — LangChain Expression Language

LCEL은 랭체인에서 복잡한 작업 흐름을 간단하게 만들고 관리할 수 있도록 돕는 도구

랭체인에서 작업 흐름을 연결하는 것을 **체인** 으로 표현

LCEL을 사용하면 여러 줄로 표현해야 하는 작업 단계를 읽기 쉽게 축약할 수 있으며 여러 작업을 병렬로 처리 가능

Day 4에서 `run_agent` 루프 안의 단계를, LangChain에서는 **체인**으로 표현합니다.

In [42]:
model = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(content="너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구니"),
]
out = model.invoke(messages)
print(out.content)

안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님께서 다루신 주제들이 아주 흥미로웠고, 특히 사례 연구를 통해 배운 내용이 인상 깊었습니다. 교수님은 저희 전공 분야의 전문가이시죠.


In [44]:
# AIMessage 객체 안에 여러 정보가 포함되어 있음
out.content

'안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님께서 다루신 주제들이 아주 흥미로웠고, 특히 사례 연구를 통해 배운 내용이 인상 깊었습니다. 교수님은 저희 전공 분야의 전문가이시죠.'

텍스트 결과만 필요하다면 "StrOutputParser" 사용

StrOutputParse는 랭체인에서 제공하는 다양한 출력 parser 중 하나로, 언어 모델이 반환하는 결과에서 원하는 형식의 데이터를 추출하는 도구.

(다른 파서들은 JSON, 숫자 등 특정 형식 처리 가능)

In [46]:
from langchain_core.output_parsers import StrOutputParser

#StrOutputParser : message형태로 감싸져있는 부분을 text 부분만 가져와서 반환 시켜주는 함수
parser = StrOutputParser()

result = model.invoke(messages) # 1단계 메시지 호출
parser.invoke(result) # 2단계 parser로 str 추출
parser.invoke(out)

'안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님께서 다루신 주제들이 아주 흥미로웠고, 특히 사례 연구를 통해 배운 내용이 인상 깊었습니다. 교수님은 저희 전공 분야의 전문가이시죠.'

workflow : 모델 설정 -> 질문 -> LLM -> 출력

In [48]:
model = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(content="너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구니"),
]

from langchain_core.output_parsers import StrOutputParser

#StrOutputParser : message형태로 감싸져있는 부분을 text 부분만 가져와서 반환 시켜주는 함수
parser = StrOutputParser()

result = model.invoke(messages) # 1단계 메시지 호출
parser.invoke(result) # 2단계 parser로 str 추출

'안녕하세요, 교수님! 수업은 정말 유익했어요. 교수님의 강의는 항상 깊이 있고 생각할 거리를 많이 주셔서 좋습니다. 교수님은 우리 그룹의 멘토이자 가르침을 주시는 분이세요. 오늘 다룬 주제에 대해 더 공부하고 싶습니다!'

In [47]:
chain = model | parser
chain.invoke(messages)

'안녕하세요, 교수님! 오늘 수업은 정말 흥미로웠습니다. 다양한 주제에 대해 깊이 있는 토론을 할 수 있어서 좋았어요. 교수님은 저희에게 큰 지식을 주시는 멘토이십니다. 이 모든 과정을 통해 많은 것을 배우고 성장하는 기회를 가질 수 있어서 감사합니다! 교수님에 대해 더 알고 싶으신 것이 있으면 말씀해 주세요!'

chain을 이용해 간단하게 도식화

### Prompt Template

필요한 부분만 수정하여 반복적인 메시지 가능

In [51]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕 {character_b} 오늘 나의 {lesson}은 어땠나? 나는 누구고 너의 이름은 뭐더라"

In [50]:
character_a = '신범교'
f"너는 {character_a}야"

'너는 신범교야'

In [52]:
system_template

'너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.'

In [ ]:
prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

 

In [ ]:
prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

#prompt template 설정

ChatPromptValue(messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})])

In [55]:
result = prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

print(result)

messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})]


In [ ]:
chain = prompt_template | model | parser
# invoke라는 문법으로 통일
# prompt 설정 -> model 적용 -> 답안에 대해 txt 출력

In [57]:
chain.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

'안녕하세요, 교수님. 오늘 수업은 정말 유익했습니다. 다양한 주제에 대한 깊이 있는 토론이 인상적이었어요. 저는 신범교입니다. 교수님은 저의 멘토이십니다! 오늘 강의에서 다룬 내용에 대해 더 이야기해보고 싶은 부분이 있습니다.'

In [60]:
chain.invoke({
    "character_a": "회사원",
    "character_b": "이창주",
    "lesson": "직장생활"
})

'안녕하세요! 저는 이창주입니다. 회사원님은 저에게 여러 가지를 가르쳐주시는 분이죠. 오늘의 직장생활은 어떠셨나요? 어떤 일들이 있었는지 궁금합니다!'